# CSCI 61 Cryptography Hands-On

Companion to slides `07_cryptography_hands_on`. Run the cells in order; later cells use names from earlier ones.

Requires Python 3.9+ and `pip install cryptography pillow`.

### Setup

In [ ]:
# pip install cryptography
import os, hashlib, hmac, secrets
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
print("ready")

## Part 1: Classical ciphers and XOR

### XOR: encrypt and decrypt with the same key

In [ ]:
def xor(data: bytes, key: bytes) -> bytes:
    return bytes(d ^ k for d, k in zip(data, key))

msg = b"HI"
key = bytes([0b01011010, 0b11110000])
ct = xor(msg, key)
print(ct.hex())           # ciphertext
print(xor(ct, key))       # XOR again -> plaintext

### Caesar cipher

In [ ]:
import string
A = string.ascii_uppercase

def caesar(text, shift):
    return "".join(A[(A.index(c) + shift) % 26]
                   if c in A else c for c in text)

ct = caesar("ATTACK AT DAWN", 3)
print(ct)
print(caesar(ct, -3))

### Brute force: try every key

In [ ]:
ct = "WKH NHB LV XQGHU WKH PDW"
for shift in range(26):
    guess = caesar(ct, -shift)
    if "THE" in guess.split():
        print(shift, guess)

### Frequency analysis

In [ ]:
from collections import Counter
ct = caesar("MEET ME NEAR THE OLD TREE BESIDE THE "
            "RIVER AT SEVEN THIS EVENING", 7)
counts = Counter(c for c in ct if c.isalpha())
print(counts.most_common(4))
top = counts.most_common(1)[0][0]
shift = (A.index(top) - A.index("E")) % 26
print(shift, caesar(ct, -shift))

### Reusing a one-time pad key

In [ ]:
key = os.urandom(16)
c1 = xor(b"PAY ALICE 100 PHP", key)
c2 = xor(b"PAY BOB 9000 PHP!", key)
leak = xor(c1, c2)                 # the key cancels out
print(xor(leak, b"PAY ALICE 100 PHP"))

## Part 2: Block ciphers

### AES works on 16-byte blocks

In [ ]:
from cryptography.hazmat.primitives.ciphers import (
    Cipher, algorithms, modes)

key = os.urandom(32)                       # AES-256
block = b"SIXTEEN BYTE MSG"                # exactly 16 bytes
enc = Cipher(algorithms.AES(key), modes.ECB()).encryptor()
print(len(block), enc.update(block).hex())

### ECB: same block in, same block out

In [ ]:
enc = Cipher(algorithms.AES(key), modes.ECB()).encryptor()
ct = enc.update(b"YELLOW SUBMARINE" * 3)
for i in range(0, len(ct), 16):
    print(i // 16, ct[i:i+16].hex())

### CBC: chain each block to the one before

In [ ]:
iv = os.urandom(16)                      # new IV every message
enc = Cipher(algorithms.AES(key), modes.CBC(iv)).encryptor()
ct = enc.update(b"YELLOW SUBMARINE" * 3) + enc.finalize()
for i in range(0, len(ct), 16):
    print(i // 16, ct[i:i+16].hex())

### Exercise: encrypt an image with ECB and CBC

Encrypt the raw pixels of an image and look at the result.

In [ ]:
from PIL import Image

def encrypt_image(path, mode):
    im = Image.open(path).convert("RGB")
    raw = im.tobytes()
    enc = Cipher(algorithms.AES(key), mode).encryptor()
    ct = enc.update(raw + b"\0" * (-len(raw) % 16)) + enc.finalize()
    return Image.frombytes("RGB", im.size, ct[:len(raw)])

# encrypt_image("logo.png", modes.ECB()).save("ecb.png")
# encrypt_image("logo.png", modes.CBC(os.urandom(16))).save("cbc.png")

### PKCS#7 padding

In [ ]:
from cryptography.hazmat.primitives import padding

padder = padding.PKCS7(128).padder()       # 128-bit blocks
padded = padder.update(b"HELLO CSCI 61") + padder.finalize()
print(len(padded), padded)

### CTR: turn a block cipher into a stream cipher

In [ ]:
nonce = os.urandom(16)
enc = Cipher(algorithms.AES(key), modes.CTR(nonce)).encryptor()
ct = enc.update(b"any length, no padding")
print(len(ct), ct.hex())
dec = Cipher(algorithms.AES(key), modes.CTR(nonce)).decryptor()
print(dec.update(ct))

### AES-GCM: encryption with a built-in integrity tag

In [ ]:
key = AESGCM.generate_key(bit_length=256)
aes = AESGCM(key)
nonce = os.urandom(12)                  # 96-bit, never reuse
ct = aes.encrypt(nonce, b"grades: A, B+, A-", b"csci61")
print(len(ct), "bytes = 17 plaintext + 16 tag")
print(aes.decrypt(nonce, ct, b"csci61"))

### Tampering is detected

In [ ]:
from cryptography.exceptions import InvalidTag

bad = bytearray(ct)
bad[8] ^= 0x01                         # flip one bit
try:
    aes.decrypt(nonce, bytes(bad), b"csci61")
except InvalidTag:
    print("InvalidTag: message was changed")

## Part 3: Hashing and integrity

### SHA-256: any input, 256-bit digest

In [ ]:
for m in [b"", b"a", b"CSCI 61", b"x" * 1_000_000]:
    d = hashlib.sha256(m).hexdigest()
    print(f"{len(m):>7} bytes -> {d[:32]}...")

### The avalanche effect

In [ ]:
h1 = hashlib.sha256(b"Transfer 100 PHP").digest()
h2 = hashlib.sha256(b"Transfer 900 PHP").digest()
diff = sum(bin(a ^ b).count("1") for a, b in zip(h1, h2))
print(h1.hex()[:32])
print(h2.hex()[:32])
print(diff, "of 256 bits changed")

### Check a download against its published hash

In [ ]:
def sha256_file(path, chunk=65536):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(chunk):
            h.update(block)
    return h.hexdigest()

with open("notes.txt", "w") as f:
    f.write("CSCI 61 notes\n")
print(sha256_file("notes.txt"))

### HMAC: a hash only the key holder can make

In [ ]:
k = secrets.token_bytes(32)                     # shared secret
msg = b"amount=100&to=alice"
tag = hmac.new(k, msg, hashlib.sha256).hexdigest()
print(tag)

def verify(k, msg, tag):
    good = hmac.new(k, msg, hashlib.sha256).hexdigest()
    return hmac.compare_digest(good, tag)      # constant time
print(verify(k, msg, tag), verify(k, b"amount=900&to=alice", tag))

### Passwords: salt and a slow hash

In [ ]:
pw = b"correct horse battery staple"
salt = os.urandom(16)                     # unique per user
h = hashlib.scrypt(pw, salt=salt, n=2**14, r=8, p=1, dklen=32)
print(salt.hex(), h.hex()[:32])

# same password, different salt -> different hash
h2 = hashlib.scrypt(pw, salt=os.urandom(16), n=2**14, r=8, p=1)
print(h.hex()[:32] == h2.hex()[:32])

### How fast can an attacker guess?

In [ ]:
import time
def rate(fn, seconds=0.5):
    n, t = 0, time.perf_counter()
    while time.perf_counter() - t < seconds:
        fn(); n += 1
    return n / seconds
print("sha256 :", round(rate(lambda: hashlib.sha256(pw).digest())))
print("scrypt :", round(rate(lambda: hashlib.scrypt(
      pw, salt=salt, n=2**14, r=8, p=1))))

## Part 4: Public-key cryptography

### RSA by hand

In [ ]:
p, q = 61, 53
n = p * q                    # 3233, public
phi = (p - 1) * (q - 1)      # 3120, secret
e = 17                       # public exponent
d = pow(e, -1, phi)          # 2753, private exponent
m = 65
c = pow(m, e, n)             # encrypt with (e, n)
print(n, d, c, pow(c, d, n)) # decrypt with (d, n)

### RSA with OAEP padding

In [ ]:
from cryptography.hazmat.primitives.asymmetric import rsa, padding as ap
from cryptography.hazmat.primitives import hashes

priv = rsa.generate_private_key(public_exponent=65537, key_size=3072)
pub = priv.public_key()
oaep = ap.OAEP(mgf=ap.MGF1(hashes.SHA256()),
               algorithm=hashes.SHA256(), label=None)
ct = pub.encrypt(b"meet at the library", oaep)
print(len(ct), priv.decrypt(ct, oaep))

### Digital signatures

In [ ]:
from cryptography.exceptions import InvalidSignature
pss = ap.PSS(mgf=ap.MGF1(hashes.SHA256()),
             salt_length=ap.PSS.MAX_LENGTH)
doc = b"Final grade: A"
sig = priv.sign(doc, pss, hashes.SHA256())
pub.verify(sig, doc, pss, hashes.SHA256())        # silent = valid
try:
    pub.verify(sig, b"Final grade: F", pss, hashes.SHA256())
except InvalidSignature:
    print("InvalidSignature: document was changed")

### Hybrid encryption: RSA carries an AES key

In [ ]:
# sender
session_key = AESGCM.generate_key(bit_length=256)
nonce = os.urandom(12)
body = AESGCM(session_key).encrypt(nonce, b"a long report..." * 1000, None)
wrapped = pub.encrypt(session_key, oaep)       # only 384 bytes
# receiver
k = priv.decrypt(wrapped, oaep)
print(len(body), len(wrapped), AESGCM(k).decrypt(nonce, body, None)[:16])

## Part 5: Key exchange

### Diffie-Hellman by hand

In [ ]:
p, g = 23, 5                  # public: prime and generator
a, b = 6, 15                  # Alice's and Bob's secrets
A = pow(g, a, p)              # Alice sends 8
B = pow(g, b, p)              # Bob sends 19
print(A, B)
print(pow(B, a, p), pow(A, b, p))  # same 2

### X25519 key agreement

In [ ]:
from cryptography.hazmat.primitives.asymmetric.x25519 import X25519PrivateKey
from cryptography.hazmat.primitives.kdf.hkdf import HKDF

alice, bob = X25519PrivateKey.generate(), X25519PrivateKey.generate()
s1 = alice.exchange(bob.public_key())       # Alice uses Bob's public key
s2 = bob.exchange(alice.public_key())       # Bob uses Alice's public key
kdf = lambda s: HKDF(hashes.SHA256(), 32, None, b"csci61 chat").derive(s)
print(s1 == s2, kdf(s1).hex()[:32])

## Part 6: PKI and key management

### Make a CA and sign a certificate

In [ ]:
from cryptography import x509
from cryptography.x509.oid import NameOID
from cryptography.hazmat.primitives.asymmetric import ec
import datetime as dt

def name(cn): return x509.Name([x509.NameAttribute(NameOID.COMMON_NAME, cn)])
now = dt.datetime.now(dt.timezone.utc)
ca_key, site_key = ec.generate_private_key(ec.SECP256R1()), \
                   ec.generate_private_key(ec.SECP256R1())
cert = (x509.CertificateBuilder()
        .subject_name(name("csci61.example")).issuer_name(name("CSCI 61 Root CA"))
        .public_key(site_key.public_key()).serial_number(x509.random_serial_number())
        .not_valid_before(now).not_valid_after(now + dt.timedelta(days=90))
        .sign(ca_key, hashes.SHA256()))

### Read and verify the certificate

In [ ]:
print(cert.subject.rfc4514_string())
print(cert.issuer.rfc4514_string())
print(cert.not_valid_after_utc.date())

# the CA's public key checks the CA's signature on the certificate
ca_key.public_key().verify(cert.signature, cert.tbs_certificate_bytes,
                           ec.ECDSA(cert.signature_hash_algorithm))
print("signature OK")

### In a terminal: a real site's certificate

```sh
openssl s_client -connect www.ateneo.edu:443 -servername www.ateneo.edu </dev/null | openssl x509 -noout -subject -issuer -dates
```

### Randomness: random vs secrets

In [ ]:
import random, secrets
random.seed(2026)
print([random.randint(0, 99) for _ in range(5)])
random.seed(2026)                           # same seed, same "key"
print([random.randint(0, 99) for _ in range(5)])

print(secrets.token_hex(16))                # from the OS CSPRNG
print(secrets.token_urlsafe(16))

### A key from a password

In [ ]:
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC

def key_from_password(password: bytes, salt: bytes) -> bytes:
    kdf = PBKDF2HMAC(algorithm=hashes.SHA256(), length=32,
                     salt=salt, iterations=600_000)
    return kdf.derive(password)

salt = os.urandom(16)                  # store salt next to ciphertext
k = key_from_password(b"hunter2", salt)
print(len(k), k.hex()[:32])

## Seatwork: encrypt and sign

1. Make RSA key pairs for Alice and Bob.
2. Alice signs a message with her private key (PSS), then encrypts message + signature to Bob with hybrid encryption.
3. Bob decrypts, then verifies with Alice's public key.
4. Change one byte of the ciphertext. What error do you get, and from which step?

In [ ]:
# your code here